# Module-08: Guided Lab



In [ ]:
# Install scikit-learn: a popular library for machine learning (classification, regression, clustering, etc.)
!pip install scikit-learn
# Install gensim: a library for working with text data (topic modeling, word embeddings, similarity between documents, etc.)
!pip install gensim

This code shows how to perform Latent Semantic Analysis (LSA) to find hidden topics in a small set of sentences. It first converts text into numerical values using TF-IDF, then applies TruncatedSVD to reduce the data into topics, and finally prints the most important words for each topic.

In [ ]:
# ----------------------------------------------------------
# TOPIC MODELING WITH LSA (LATENT SEMANTIC ANALYSIS)
# ----------------------------------------------------------
# In this example, we:
# 1) Turn short text sentences into numbers using TF-IDF.
# 2) Use LSA (with TruncatedSVD) to find hidden "topics" in the text.
# 3) Print out the most important words for each topic.
# ----------------------------------------------------------

# NumPy is a library for working with arrays and numbers.
import numpy as np

# TfidfVectorizer turns a collection of text documents into a TF IDF matrix.
# TF IDF stands for "Term Frequency - Inverse Document Frequency".
# It gives higher weight to words that are important in a document
# but not too common across all documents.
from sklearn.feature_extraction.text import TfidfVectorizer

# TruncatedSVD is used here to perform LSA.
# LSA reduces the number of dimensions of our TF IDF matrix
# and helps us find hidden structure or "topics" in the text.
from sklearn.decomposition import TruncatedSVD

# ----------------------------------------------------------
# 1. Create a small sample text corpus
# ----------------------------------------------------------
# A "corpus" is just a collection (list) of text documents.
# Here, each item in the list is one sentence.
corpus = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "The cat chased the dog.",
    "The dog chased the cat."
]

# ----------------------------------------------------------
# 2. Convert text into a TF IDF matrix
# ----------------------------------------------------------
# Create a TF IDF vectorizer object.
# This object will learn the vocabulary (unique words)
# and compute TF IDF values for each word in each sentence.
vectorizer = TfidfVectorizer()

# fit_transform():
# - fit: learn the vocabulary from the corpus
# - transform: create the TF IDF matrix
# X will be a 2D matrix where:
#   rows    = sentences
#   columns = words
X = vectorizer.fit_transform(corpus)

# ----------------------------------------------------------
# 3. Apply LSA using TruncatedSVD
# ----------------------------------------------------------
# n_components = 2 means we want to find 2 topics.
# random_state is set so the results are reproducible
# (you get the same numbers every time you run it).
lsa = TruncatedSVD(n_components=2, random_state=42)

# fit_transform():
# - fit: learn the topics from the TF IDF matrix
# - transform: project each sentence into the topic space
# X_reduced will have:
#   rows    = sentences
#   columns = topics
X_reduced = lsa.fit_transform(X)

# ----------------------------------------------------------
# 4. Look at the topics and their most important words
# ----------------------------------------------------------
# Get the list of all words (terms) that the vectorizer learned.
terms = vectorizer.get_feature_names_out()

# lsa.components_ is a 2D array where:
#   rows    = topics
#   columns = words
# Each number tells us how strongly a word is related to a topic.
for i, comp in enumerate(lsa.components_):
    # Pair each term with its weight for this topic.
    terms_comp = zip(terms, comp)

    # Sort words by their weight in this topic (highest weight first)
    # and keep only the top 5 words.
    sorted_terms = sorted(terms_comp, key=lambda x: x[1], reverse=True)[:5]

    print(f"Topic {i}:")
    # Print each word and how important it is for this topic.
    for term, weight in sorted_terms:
        print(f" - {term}: {weight:.4f}")


This code demonstrates how to perform topic modeling using LDA (Latent Dirichlet Allocation) in Python. It converts sentences into a bag-of-words format, trains an LDA model to discover hidden topics, assigns a topic distribution to a new sentence, and calculates a coherence score to evaluate the quality of the topics.

In [ ]:
# ----------------------------------------------------------
# TOPIC MODELING WITH LDA AND COHERENCE SCORE
# ----------------------------------------------------------
# In this example, we:
# 1) Create a small list of sentences (a text corpus).
# 2) Turn each sentence into numbers using bag of words.
# 3) Train an LDA model to find hidden topics in the text.
# 4) Check which topic a new sentence belongs to.
# 5) Calculate a "coherence score" to see how good the topics are.
# ----------------------------------------------------------

# gensim is a Python library for working with text and topic modeling.
import gensim

# corpora helps us create dictionaries and bag of words representations.
from gensim import corpora

# LdaModel is the Latent Dirichlet Allocation model for topic modeling.
from gensim.models import LdaModel

# pprint prints complex data structures in an easy to read way.
from pprint import pprint

# CoherenceModel helps us measure how "good" or "meaningful"
# the topics from our model are.
from gensim.models.coherencemodel import CoherenceModel

# ----------------------------------------------------------
# 1. Create a small sample text corpus
# ----------------------------------------------------------
# A "corpus" is a collection of documents.
# Here, each document is just one simple sentence.
corpus = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "The cat chased the dog.",
    "The dog chased the cat."
]

# ----------------------------------------------------------
# 2. Tokenize the text (split into words)
# ----------------------------------------------------------
# For each sentence:
#   - convert the sentence to lowercase
#   - split it into words using spaces
#
# Result example:
# [
#   ["the", "cat", "sat", "on", "the", "mat."],
#   ["the", "dog", "sat", "on", "the", "log."],
#   ...
# ]
#
# Note: This code does not actually remove stop words like "the" or "on".
# In a real project, you would usually remove them to improve results.
texts = [[word for word in document.lower().split()] for document in corpus]

# ----------------------------------------------------------
# 3. Create a dictionary from the texts
# ----------------------------------------------------------
# The dictionary maps each unique word to an integer id.
# Example (not exact):
#   "cat" -> 0
#   "dog" -> 1
#   "sat" -> 2
dictionary = corpora.Dictionary(texts)

# ----------------------------------------------------------
# 4. Convert each document to bag of words (BoW)
# ----------------------------------------------------------
# Bag of words format represents each document as a list of:
#   (word_id, count) pairs.
#
# Example (not exact):
#   [(0, 1), (2, 1), (3, 2)]
# This means:
#   word with id 0 appears 1 time
#   word with id 2 appears 1 time
#   word with id 3 appears 2 times
corpus_bow = [dictionary.doc2bow(text) for text in texts]

# ----------------------------------------------------------
# 5. Train the LDA topic model
# ----------------------------------------------------------
# Parameters:
#   corpus     - our bag of words documents.
#   id2word    - the dictionary that maps ids back to words.
#   num_topics - how many topics we want the model to find.
#   random_state - set this to get the same result every time.
#   passes     - how many times the model goes through the data.
lda_model = LdaModel(
    corpus=corpus_bow,
    id2word=dictionary,
    num_topics=2,
    random_state=42,
    passes=10
)

# ----------------------------------------------------------
# 6. Print the topics that the model found
# ----------------------------------------------------------
# Each topic is shown as a list of words with weights.
# The weight tells us how strongly each word is connected to that topic.
print("Topics:")
pprint(lda_model.print_topics(num_words=5))
# num_words=5 means we show the top 5 words for each topic.

# ----------------------------------------------------------
# 7. Assign topics to a new document
# ----------------------------------------------------------
# Now we create a new sentence and see which topics it belongs to.
new_doc = "The cat chased the dog."

# Convert the new sentence to lowercase and split into words,
# then turn it into bag of words using the same dictionary as before.
new_doc_bow = dictionary.doc2bow(new_doc.lower().split())

print("\nTopic Distribution for the new document:")

# get_document_topics returns a list of (topic_id, probability) pairs.
# Example:
#   [(0, 0.8), (1, 0.2)]
# This means:
#   The new document is 80 percent Topic 0 and 20 percent Topic 1.
pprint(lda_model.get_document_topics(new_doc_bow))

# ----------------------------------------------------------
# 8. Compute the coherence score of the LDA model
# ----------------------------------------------------------
# The coherence score measures how meaningful the topics are,
# based on how often the top words for each topic appear together
# in the original texts.
#
# Higher coherence usually means better topics.
coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'  # c_v is a popular coherence measure for topic models.
)

# Calculate the coherence value as a single number.
coherence_lda = coherence_model_lda.get_coherence()

# Print the coherence score.
print(f"Coherence Score: {coherence_lda}")


This code demonstrates how to perform topic modeling by converting text into bag-of-words form, training a topic model, viewing the discovered topics, assigning a topic distribution to a new document, and calculating a coherence score to evaluate how meaningful the topics are.

In [ ]:
# ---------------------------------------------------------
# TOPIC MODELING WITH HDP (Hierarchical Dirichlet Process)
# ---------------------------------------------------------
# In this example we:
# 1) Prepare a small text corpus.
# 2) Turn each document into a bag-of-words representation.
# 3) Train an HDP topic model with Gensim to discover topics.
# 4) Inspect the topics and see how a new document is assigned to them.
# 5) Compute a coherence score to see how meaningful the topics are.
# ---------------------------------------------------------

import gensim
from gensim import corpora
from gensim.models import HdpModel, CoherenceModel
from pprint import pprint



# ---------------------------------------------------------
# 1. SAMPLE TEXT CORPUS
# ---------------------------------------------------------
# Each string represents a short document.
corpus = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "The cat chased the dog.",
    "The dog chased the cat."
]

# ---------------------------------------------------------
# 2. TOKENIZE THE TEXT
# ---------------------------------------------------------
# We:
# - convert each document to lowercase
# - split on spaces to get a list of words
# (In a real project, you would also remove stop words and punctuation.)
texts = [
    [word for word in document.lower().split()]
    for document in corpus
]

# ---------------------------------------------------------
# 3. CREATE A DICTIONARY AND BAG-OF-WORDS CORPUS
# ---------------------------------------------------------
# The dictionary maps each unique word to an integer ID.
dictionary = corpora.Dictionary(texts)

# Convert each document into a "bag of words":
# a list of (word_id, count) pairs.
corpus_bow = [dictionary.doc2bow(text) for text in texts]

# ---------------------------------------------------------
# 4. TRAIN THE HDP TOPIC MODEL
# ---------------------------------------------------------
# HdpModel automatically tries to discover a suitable number of topics
# instead of requiring us to choose a fixed number.
hdp_model = HdpModel(corpus=corpus_bow, id2word=dictionary)

# ---------------------------------------------------------
# 5. PRINT THE DISCOVERED TOPICS
# ---------------------------------------------------------
print("Topics:")
# print_topics shows the top words for each topic.
# num_topics=2 means we print 2 topics.
# num_words=5 means we show the 5 most important words for each topic.
pprint(hdp_model.print_topics(num_topics=2, num_words=5))

# ---------------------------------------------------------
# 6. ASSIGN TOPICS TO A NEW DOCUMENT
# ---------------------------------------------------------
new_doc = "The cat chased the dog."

# Convert the new document to bag-of-words using the SAME dictionary.
new_doc_bow = dictionary.doc2bow(new_doc.lower().split())

print("\nTopic Distribution for the new document:")
# hdp_model[new_doc_bow] returns a list of (topic_id, probability) pairs.
pprint(hdp_model[new_doc_bow])

# ---------------------------------------------------------
# 7. COMPUTE COHERENCE SCORE
# ---------------------------------------------------------
# Coherence measures how "interpretable" or "coherent" the topics are.
# Higher coherence usually means the topics make more sense to humans.
coherence_model_hdp = CoherenceModel(
    model=hdp_model,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_hdp = coherence_model_hdp.get_coherence()
print(f"\nCoherence Score: {coherence_hdp}")
